# QUY TRÌNH 1 - THU THẬP DỮ LIỆU

## E-commerce Data Lakehouse trên Object Storage

Dataset:
Olist Brazilian E-Commerce Public Dataset

Mục tiêu:
- Thu thập dữ liệu Olist
- Lựa chọn các bảng cần thiết
- Kiểm tra dữ liệu ban đầu
- Chuẩn hóa tên file
- Lưu dữ liệu Raw

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


In [5]:
# Xác định đường dẫn đến thư mục gốc của dự án và các thư mục dữ liệu
BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

SOURCE_DIR = BASE_DIR / "data" / "source"
RAW_DIR = BASE_DIR / "data" / "raw"

print("BASE_DIR :", BASE_DIR)
print("SOURCE_DIR:", SOURCE_DIR)
print("RAW_DIR   :", RAW_DIR)


BASE_DIR : /Users/camlee05/development/E-commerce-Data-Lakehouse
SOURCE_DIR: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/source
RAW_DIR   : /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw


In [6]:
# Khai báo các tên file dữ liệu
DATASETS = {
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "payments": "olist_order_payments_dataset.csv"
}

customers = pd.read_csv(SOURCE_DIR / DATASETS["customers"])
products = pd.read_csv(SOURCE_DIR / DATASETS["products"])
orders = pd.read_csv(SOURCE_DIR / DATASETS["orders"])
order_items = pd.read_csv(SOURCE_DIR / DATASETS["order_items"])
reviews = pd.read_csv(SOURCE_DIR / DATASETS["reviews"])
payments = pd.read_csv(SOURCE_DIR / DATASETS["payments"])

In [7]:
# Tạo một dictionary để lưu trữ các DataFrame
datasets = {
    name: pd.read_csv(SOURCE_DIR / file_name)
    for name, file_name in DATASETS.items()
}

In [8]:
# Kiểm tra số lượng bản ghi và cột của từng dataset
overview = []

for name, df in datasets.items():
    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

overview_df = pd.DataFrame(overview)

overview_df

,dataset,rows,columns
0,customers,99441,5
1,products,32951,9
2,orders,99441,8
3,order_items,112650,7
4,reviews,99224,7
5,payments,103886,5


In [9]:
df.dtypes

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

In [10]:
# Hiển thị thông tin chi tiết về từng dataset
for name, df in datasets.items():
    print("=" * 60)
    print(f"DATASET: {name}")
    print("Số dòng:", len(df))
    print("Số cột:", len(df.columns))
    print("Các cột:")
    print(df.columns.tolist())

DATASET: customers
Số dòng: 99441
Số cột: 5
Các cột:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
DATASET: products
Số dòng: 32951
Số cột: 9
Các cột:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
DATASET: orders
Số dòng: 99441
Số cột: 8
Các cột:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
DATASET: order_items
Số dòng: 112650
Số cột: 7
Các cột:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
DATASET: reviews
Số dòng: 99224
Số cột: 7
Các cột:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [11]:
# Kiểm tra giá trị thiếu trong từng dataset
def missing_values(df):
    result = df.isnull().sum().reset_index()
    result.columns = ["Column", "Missing Values"]
    result["Missing %"] = (
        result["Missing Values"] / len(df) * 100
    ).round(2)
    
    return result[result["Missing Values"] > 0]

for name, df in datasets.items():
    print("=" * 60)
    print(f"Missing values - {name}")
    display(missing_values(df))

Missing values - customers


,Column,Missing Values,Missing %


Missing values - products


,Column,Missing Values,Missing %
1,product_category_name,610,1.85
2,product_name_lenght,610,1.85
3,product_description_lenght,610,1.85
4,product_photos_qty,610,1.85
5,product_weight_g,2,0.01
6,product_length_cm,2,0.01
7,product_height_cm,2,0.01
8,product_width_cm,2,0.01


Missing values - orders


,Column,Missing Values,Missing %
4,order_approved_at,160,0.16
5,order_delivered_carrier_date,1783,1.79
6,order_delivered_customer_date,2965,2.98


... (2 outputs left out)

Missing values - reviews


,Column,Missing Values,Missing %
3,review_comment_title,87656,88.34
4,review_comment_message,58247,58.70


Missing values - payments


,Column,Missing Values,Missing %


In [12]:
# Kiểm tra các bản ghi trùng lặp trong từng dataset
df.duplicated().sum()

np.int64(0)

In [13]:
# Kiểm tra các giá trị trùng lặp trong các cột khóa chính của từng dataset
print(
    "Duplicate customer_id:",
    customers["customer_id"].duplicated().sum()
)

print(
    "Duplicate product_id:",
    products["product_id"].duplicated().sum()
)

print(
    "Duplicate order_id:",
    orders["order_id"].duplicated().sum()
)

print(
    "Duplicate review_id:",
    reviews["review_id"].duplicated().sum()
)

Duplicate customer_id: 0
Duplicate product_id: 0
Duplicate order_id: 0
Duplicate review_id: 814


In [14]:
# Kiểm tra các cột khóa chính của từng dataset
key_columns = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "reviews": "review_id",
}

pk_results = []

for name, column in key_columns.items():
    df = datasets[name]

    null_count = int(df[column].isna().sum())
    duplicate_count = int(df[column].duplicated().sum())

    pk_results.append({
        "dataset": name,
        "primary_key": column,
        "null_count": null_count,
        "duplicate_count": duplicate_count,
        "valid": (
            null_count == 0 and
            duplicate_count == 0
        )
    })

pd.DataFrame(pk_results)

,dataset,primary_key,null_count,duplicate_count,valid
0,customers,customer_id,0,0,True
1,products,product_id,0,0,True
2,orders,order_id,0,0,True
3,reviews,review_id,0,814,False


In [15]:
# Kiểm tra quan hệ giữa các bảng (foreign key relationships)
def check_foreign_key(
    child_df,
    parent_df,
    key_col
):
    parent_ids = set(
        parent_df[key_col].dropna()
    )

    invalid_mask = (
        child_df[key_col].notna()
        & ~child_df[key_col].isin(parent_ids)
    )

    invalid_rows = int(invalid_mask.sum())

    invalid_keys = int(
        child_df.loc[
            invalid_mask,
            key_col
        ].nunique()
    )

    return {
        "invalid_rows": invalid_rows,
        "invalid_unique_keys": invalid_keys
    }

check_foreign_key(
    datasets["orders"],
    datasets["customers"],
    "customer_id"
)


{'invalid_rows': 0, 'invalid_unique_keys': 0}

In [16]:
# Lưu dữ liệu đã ingest vào thư mục raw
raw_datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
    "payments": payments
}

for name, df in raw_datasets.items():
    output_path = RAW_DIR / f"{name}.csv"
    df.to_csv(output_path, index=False)
    

    print(f"✓ Đã lưu: {output_path}")

✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/customers.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/products.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/orders.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/order_items.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/reviews.csv
✓ Đã lưu: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw/payments.csv


In [17]:
# Kiểm tra các file đã được lưu trong thư mục raw
print("Các file trong Raw:")

for file in RAW_DIR.iterdir():
    print("-", file.name)

Các file trong Raw:
- customers.csv
- products.csv
- reviews.csv
- orders.csv
- payments.csv
- order_items.csv
